In [2]:
import numpy as np
import pandas as pd

### 1. In the above dataset

##### a. In case age is less than 18, replace it with mean of age values.

In [3]:
# Load the acquisition data
ca = pd.read_csv('Customer Acquisition.csv')
ca.head()

,No,Customer,Age,City,Product,Limit,Company,Segment
0,1,A1,76,BANGALORE,Gold,500000.0,C1,Self Employed
1,2,A2,71,CALCUTTA,Silver,100000.0,C2,Salaried_MNC
2,3,A3,34,COCHIN,Platimum,10000.0,C3,Salaried_Pvt
3,4,A4,47,BOMBAY,Platimum,10001.0,C4,Govt
4,5,A5,56,BANGALORE,Platimum,10002.0,C5,Normal Salary


In [4]:
# Step 1: Calculate mean age BEFORE any replacement (uses original, unfixed values)
mean_age = ca['Age'].mean()
print(f'Mean age (before fixing): {mean_age}')

Mean age (before fixing): 46.49


In [5]:
# Step 2: Replace any Age < 18 with that mean value
ca.loc[ca['Age'] < 18, 'Age'] = mean_age

C:\Users\khush\AppData\Local\Temp\ipykernel_23020\2690506320.py:2: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '46.49' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  ca.loc[ca['Age'] < 18, 'Age'] = mean_age


In [6]:
# Step 3: Sanity check - confirm no age below 18 remains
print(f'Any age still below 18?: {(ca['Age'] < 18).any()}\n')
print(f'Five Number Summary: \n{ca['Age'].describe()}')

Any age still below 18?: False

Five Number Summary: 
count    100.000000
mean      48.399400
std       16.402368
min       19.000000
25%       35.000000
50%       46.745000
75%       60.250000
max       79.000000
Name: Age, dtype: float64


##### b. In case spend amount is more than the limit, replace it with 50% of that customer’s limit. (customer’s limit provided in acquisition table is the per transaction limit on his card)

In [7]:
# Load the spend table
sp = pd.read_csv('spend.csv')
sp.head()

,Sl No:,Customer,Month,Type,Amount
0,1,A1,12-Jan-04,JEWELLERY,485470.80
1,2,A1,3-Jan-04,PETRO,410556.13
2,3,A1,15-Jan-04,CLOTHES,23740.46
3,4,A1,25-Jan-04,FOOD,484342.47
4,5,A1,17-Jan-05,CAMERA,369694.07


In [8]:
# Step 1: Bring Limit into the Spend table by merging on customers
# how='left' -> keep all rows from spend.csv, just attach Limit where a match exists.
sp = sp.merge(ca[['Customer', 'Limit']], on='Customer', how='left') 

In [9]:
# Step 2: Whenever Amount > Limit, replace Amount with 50% of that row's limit
sp.loc[sp['Amount'] > sp['Limit'], 'Amount'] = 0.5 * sp['Limit']

In [10]:
# Sanity check: no spend should now exceed the limit
print(f'Any spend still above limit: {(sp['Amount'] > sp['Limit']).any()}')

Any spend still above limit: False


##### c. In case the repayment amount is more than the limit, replace the repayment with the limit.

In [11]:
# Load the Repayment table
rp = pd.read_csv('Repayment.csv')
rp.head()

,SL No:,Customer,Month,Amount,Unnamed: 4
0,NaN,A1,12-Jan-04,495414.75,NaN
1,2.0,A1,3-Jan-04,245899.02,NaN
2,3.0,A1,15-Jan-04,259490.06,NaN
3,4.0,A1,25-Jan-04,437555.12,NaN
4,5.0,A1,17-Jan-05,165972.88,NaN


In [12]:
# Step 1: Bring Limit into Repayment table by merging on customers
rp = rp.merge(ca[['Customer', 'Limit']], on='Customer', how='left')

In [13]:
# Step 2: Whenever Amount > Limit, replace Amount with limit
rp.loc[rp['Amount'] > rp['Limit'], 'Amount'] = rp['Limit']

In [14]:
# Sanity check: no repayment should exceed the limit
print(f'Any repayment should above the limit: {(rp['Amount'] > rp['Limit']).any()}')

Any repayment should above the limit: False


### 2. From the above dataset create the following summaries:

##### a. How many distinct customers exist?

In [15]:
# use nunique() method to find the distinct customers
num_distinct_customers = sp['Customer'].nunique()
num_distinct_customers

100

##### b. How many distinct categories exist?

In [16]:
# Product categories unique value count
num_of_distinct_product_categories = ca['Product'].nunique()
num_of_distinct_product_categories

3

In [17]:
# Type categories unique value count
num_of_distinct_type_categories = sp['Type'].nunique()
num_of_distinct_type_categories

15

##### c. Average Monthly Spend by Customers

**Assumption:** "Average monthly spend by customer" is interpreted as the average total spend across all customers, per month (bank-wide monthly average) - not the average spend of each individual customer.

In [18]:
# Convert the Month column (currently plain text like '12-Jun-04) into a real datetime. 
sp['Month'] = pd.to_datetime(sp['Month'], format='%d-%b-%y')

# Sanity check
print(sp[['Month']].head(), "\n")
print(sp['Month'].dtype)

       Month
0 2004-01-12
1 2004-01-03
2 2004-01-15
3 2004-01-25
4 2005-01-17 

datetime64[ns]


In [19]:
# Step 1: Creating a new column for each calendar month
sp['Month_Period'] = sp['Month'].dt.to_period('M')

In [20]:
# Step 2: Total spend across ALL customers, for each calendar month
monthly_total_spend = sp.groupby('Month_Period')['Amount'].sum()

In [21]:
# Step 3: Average of those monthly totals
average_monthly_spend = monthly_total_spend.mean()
print(f'Average monthly spend (bank-wide): {average_monthly_spend:.2f}')

Average monthly spend (bank-wide): 7326036.17


##### d. What is the average monthly repayment by customers?

In [22]:
# Drop the 23 full-blank rows first. 
rp = rp.dropna(how='all')

# Drop the junk empty column
rp = rp.drop(columns='Unnamed: 4')

In [23]:
# Convert the month in correct format
rp['Month'] = pd.to_datetime(sp['Month'], format='%d-%b-%y')

In [24]:
# Step 1: Create a new column name monthly_period, for each calendar month
rp['Month_Period'] = rp['Month'].dt.to_period('M')

In [25]:
# Step 2: Total repayment across all customers, for each calendar month
monthly_total_repayment = rp.groupby('Month_Period')['Amount'].sum()

In [26]:
# Step 3: Average of those monthly totals
average_monthly_repayment = monthly_total_repayment.mean()

print(f'Average monthly repayment: {average_monthly_repayment:.2f}')

Average monthly repayment: 8166078.67


##### e. If the monthly rate of interest is 2.9%, what is the profit for the bank for each month? 

In [27]:
# Step 1: Total spend per calendar month 
monthly_spend = sp.groupby('Month_Period')['Amount'].sum()

In [28]:
# Step 2: Total repayment per calendar month
monthly_repayment = rp.groupby('Month_Period')['Amount'].sum()

In [29]:
# Step 3: Monthly Profit = Repayment - Spend, aligned by Month_Period
monthly_profit = monthly_repayment - monthly_spend

In [30]:
# Step 4: Clip negative profits to 0 (interest only on positive profit)
monthly_profit_clipped = monthly_profit.clip(lower=0)

In [31]:
# Step 5: Apply 2.9% monthly interest rate
monthly_interest_profit = monthly_profit_clipped * 0.029

print(f'Monthly Interest Profit: {monthly_interest_profit}')

Monthly Interest Profit: Month_Period
2004-01    181081.12923
2004-02     39217.85913
2004-03     13904.04826
2004-04     23594.80977
2004-05      4782.30822
2004-09         0.00000
2004-11     33284.60094
2005-01         0.00000
2005-02    179753.26302
2005-04         0.00000
2005-05     17092.99875
2005-06         0.00000
2005-07      2001.68904
2005-08      3288.12063
2005-09         0.00000
2005-10     25276.42436
2005-11         0.00000
2005-12     46958.18185
2006-01     11220.59532
2006-02      4154.00698
2006-03    138288.40182
2006-04     51594.11402
2006-05     54845.02127
2006-06     16029.45913
2006-07         0.00000
2006-08      9918.47415
2006-09     32773.04616
2006-10     19066.45426
2006-11     19249.87955
2006-12         0.00000
Freq: M, Name: Amount, dtype: float64


##### f. What are the top 5 product types?

In [32]:
type_totals = sp.groupby('Type')['Amount'].sum()
top_5_types = type_totals.sort_values(ascending=False).head(5)
top_5_types

Type
PETRO           28597384.98
CAMERA          27690738.44
FOOD            20519243.60
AIR TICKET      20155847.12
TRAIN TICKET    19995825.72
Name: Amount, dtype: float64

##### g. Which city is having maximum spend?

In [33]:
# Bring city into the Spend table 
new_sp = sp.merge(ca[['Customer', 'City']], on='Customer', how='left')

In [34]:
# Total spend per city
city_total = new_sp.groupby('City')['Amount'].sum()

In [35]:
# Find the city with the Maximum total spend
max_spend_city = city_total.idxmax()
max_spend_value = city_total.max()

print(f'--- City with maximum spend ---\n{max_spend_city} = {max_spend_value}')

--- City with maximum spend ---
COCHIN = 45963513.5


##### h. Which age group is spending more money?

In [36]:
# Step 1: Merge the Age from ca table in sp table. 
new_sp = sp.merge(ca[['Customer', 'Age']], on='Customer', how='left')

In [38]:
new_sp[new_sp['Age'] < 18]

,Sl No:,Customer,Month,Type,Amount,Limit,Month_Period,Age
